# 6.2. 图像卷积

本节从二维互相关开始，实现一个简单的卷积层，并观察卷积核如何检测图像边缘。

### 6.2.1 互相关运算

卷积窗口从左上角开始滑动，对窗口内元素与核逐元素相乘后求和。请实现 `corr2d(X, K)`：输入二维张量 `X` 和卷积核 `K`，返回互相关结果。然后用 3×3 输入和 2×2 核验证输出应为 `[[19, 25], [37, 43]]`。

In [42]:
import torch
from torch import nn

def corr2d(X,K):
    h, w = K.shape
    Y = torch.zeros(
        (X.shape[0] - h + 1, X.shape[1] - w + 1)
    )
    for i in range(Y.shape[0]):
        for j in range(Y.shape[1]):
            Y[i,j]=(X[i:i+h,j:j+w]*K).sum()
    return Y
X = torch.tensor([[0.0, 1.0, 2.0], [3.0, 4.0, 5.0], [6.0, 7.0, 8.0]])
K = torch.tensor([[0.0, 1.0], [2.0, 3.0]])
corr2d(X, K)

tensor([[19., 25.],
        [37., 43.]])

### 6.2.2 卷积层

继承 `nn.Module` 实现 `Conv2D`，包含可学习的权重和偏置，并在 `forward` 中调用二维互相关。使用 `nn.Parameter` 定义参数，之后用一个简单样本检查输出形状。

In [46]:
class Conv2D(nn.Module):
    def __init__(self,kernel_size):
        super().__init__()
        self.weight = nn.Parameter(torch.rand(kernel_size))
        self.bias = nn.Parameter(torch.zeros(1))
    def forward(self,X):
        Y=corr2d(X,self.weight)+self.bias
        return Y

conv2d=Conv2D([3,3])
print(conv2d(X))
        

tensor([[24.6868]], grad_fn=<AddBackward0>)


### 6.2.3 图像中的目标边缘检测

构造一个水平线条图像和核 `[[1, -1]]`，观察卷积结果。比较原图、转置后的图像和输出，解释该卷积核为什么能够识别垂直边缘。

In [51]:
X=torch.Tensor([[1,1,0],[1,1,0],[1,1,0]])
K=torch.Tensor([[1,-1]])
print(corr2d(X,K))

tensor([[0., 1.],
        [0., 1.],
        [0., 1.]])


### 6.2.4 学习卷积核

用 `nn.Conv2d` 定义单输入、单输出通道的卷积层。以边缘图像为训练数据，使用平方损失和梯度下降学习一个能复现目标边缘的卷积核，并打印训练后的权重。

In [ ]:
conv2d=nn.Conv2d(1,1,kernel_size=(1,2),bias=False)
X = torch.ones((6, 8))
X[:, 2:6] = 0
Y = corr2d(X, K)
# 这个二维卷积层使用四维输入和输出格式（批量大小、通道、高度、宽度），
# 其中批量大小和通道数都为1
X = X.reshape((1, 1, 6, 8))
Y = Y.reshape((1, 1, 6, 7))
lr = 3e-2  # 学习率
for i in range(50):
    Y_hat=conv2d(X)
    l=(Y_hat-Y)**2
    conv2d.zero_grad()
    l.sum().backward()
    conv2d.weight.data-=lr*conv2d.weight.grad
conv2d.weight.data.reshape((1, 2))


tensor([[ 1.0000, -1.0000]])

### 6.2.5 互相关和卷积

深度学习框架通常实现的是互相关，而数学上的卷积需要先将卷积核水平、垂直翻转。对于可学习的卷积核，翻转可以被吸收到参数中，因此二者在模型表达能力上等价。

### 6.2.6 特征映射和感受野

卷积输出中的每个元素对应输入中的一个局部区域，该区域称为感受野。堆叠多层卷积会逐渐扩大感受野，使高层特征能够整合更大范围的信息。

### 6.2.7 小结与练习

练习：修改卷积核大小；用不同初始化训练边缘检测器；计算多层卷积后某个输出元素的感受野大小。